In [1]:
import os
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

# 1
### 1.1 Paths

In [2]:
step1_dir = Path("./results_step1")
step2_dir = Path("./results_step2")
step3_dir = Path("./results_step3")

step3_dir.mkdir(parents=True, exist_ok=True)

load_pair_splits = True

print("step1_dir:", step1_dir.resolve())
print("step2_dir:", step2_dir.resolve())
print("step3_dir:", step3_dir.resolve())
print("load_pair_splits:", load_pair_splits)

step1_dir: C:\Users\David\Documents\github\4530\ACIT4530\David\results_step1
step2_dir: C:\Users\David\Documents\github\4530\ACIT4530\David\results_step2
step3_dir: C:\Users\David\Documents\github\4530\ACIT4530\David\results_step3
load_pair_splits: True


### 1.2 Audit required files

In [3]:
required_step1_files = [
    "step1_config.json",
    "movies.parquet",
    "users.parquet",
    "ratings_filtered.parquet",
    "train_split_model_universe.parquet",
    "validation_split_model_universe.parquet",
    "test_split_model_universe.parquet",
]

if load_pair_splits:
    required_step1_files += [
        "train_pairs.parquet",
        "validation_pairs.parquet",
        "test_pairs.parquet",
    ]

required_step2_files = [
    "final_step2_config.json",
    "final_pipeline_audit.json",
    "user_embeddings_final.parquet",
    "item_embeddings_final.parquet",
    "user_embeddings_p_u.npy",
    "item_embeddings_q_i.npy",
    "item_bias.npy",
    "user_index_mapping.parquet",
    "item_index_mapping.parquet",
    "final_pairwise_evaluation.parquet",
    "final_generalization_summary.parquet",
    "embedding_diagnostics_summary.parquet",
]

optional_step2_files = [
    "step1_config_used.json",
    "step2_run_manifest.json",
    "final_selected_config_summary.parquet",
    "final_train_pairs_used_by_bpr_summary.parquet",
    "final_training_history.parquet",
    "pair_sampling_decision_summary.parquet",
    "hyperparameter_impact_summary.parquet",
]


def make_file_audit_row(base_dir, filename, required=True):
    file_path = base_dir / filename
    exists = file_path.exists()

    return {
        "base_dir": str(base_dir),
        "file": filename,
        "required": required,
        "exists": exists,
        "size_mb": round(file_path.stat().st_size / (1024 ** 2), 4) if exists else None,
    }


file_audit_rows = []

for filename in required_step1_files:
    file_audit_rows.append(
        make_file_audit_row(step1_dir, filename, required=True)
    )

for filename in required_step2_files:
    file_audit_rows.append(
        make_file_audit_row(step2_dir, filename, required=True)
    )

for filename in optional_step2_files:
    file_audit_rows.append(
        make_file_audit_row(step2_dir, filename, required=False)
    )

required_file_audit = pd.DataFrame(file_audit_rows)

display(required_file_audit)

missing_required_files = required_file_audit[
    required_file_audit["required"] & ~required_file_audit["exists"]
]

if len(missing_required_files) > 0:
    raise FileNotFoundError(
        "Missing required files:\n"
        + missing_required_files[["base_dir", "file"]].to_string(index=False)
    )

print("required file audit passed")

,base_dir,file,required,exists,size_mb
0,results_step1,step1_config.json,True,True,0.0006
1,results_step1,movies.parquet,True,True,0.0993
2,results_step1,users.parquet,True,True,0.0686
3,results_step1,ratings_filtered.parquet,True,True,8.5200
4,results_step1,train_split_model_universe.parquet,True,True,6.1756
5,results_step1,validation_split_model_universe.parquet,True,True,1.0917
6,results_step1,test_split_model_universe.parquet,True,True,1.1787
7,results_step1,train_pairs.parquet,True,True,26.6392
8,results_step1,validation_pairs.parquet,True,True,1.9347
9,results_step1,test_pairs.parquet,True,True,2.0204


required file audit passed


### 1.3 Helper functions

In [4]:
def read_json(file_path):
    with open(file_path, "r") as file:
        return json.load(file)


def load_parquet(file_path):
    return pd.read_parquet(file_path)


def standardize_pair_columns(pair_df):
    pair_df = pair_df.copy()

    rename_map = {
        "preferred_item": "pos_item",
        "less_preferred_item": "neg_item",
        "rating_preferred": "rating_pos",
        "rating_less_preferred": "rating_neg",
        "timestamp_preferred": "timestamp_pos",
        "timestamp_less_preferred": "timestamp_neg",
    }

    active_rename_map = {
        old_name: new_name
        for old_name, new_name in rename_map.items()
        if old_name in pair_df.columns
    }

    pair_df = pair_df.rename(columns=active_rename_map)

    required_columns = [
        "user_id",
        "pos_item",
        "neg_item",
        "rating_pos",
        "rating_neg",
        "rating_diff",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in pair_df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required pairwise columns after standardization: {missing_columns}"
        )

    return pair_df


def make_matrix_check(name, matrix):
    return {
        "name": name,
        "shape": tuple(matrix.shape),
        "dtype": str(matrix.dtype),
        "has_nan": bool(np.isnan(matrix).any()),
        "has_inf": bool(np.isinf(matrix).any()),
        "min": float(np.min(matrix)),
        "max": float(np.max(matrix)),
        "mean": float(np.mean(matrix)),
        "std": float(np.std(matrix)),
    }


def make_norm_summary(name, matrix):
    norms = np.linalg.norm(matrix, axis=1)

    return {
        "embedding": name,
        "rows": matrix.shape[0],
        "dimensions": matrix.shape[1],
        "mean_l2_norm": float(norms.mean()),
        "median_l2_norm": float(np.median(norms)),
        "std_l2_norm": float(norms.std()),
        "min_l2_norm": float(norms.min()),
        "p01_l2_norm": float(np.quantile(norms, 0.01)),
        "p05_l2_norm": float(np.quantile(norms, 0.05)),
        "p95_l2_norm": float(np.quantile(norms, 0.95)),
        "p99_l2_norm": float(np.quantile(norms, 0.99)),
        "max_l2_norm": float(norms.max()),
        "zero_norm_rows": int((norms == 0).sum()),
    }


def count_semantic_duplicates(pair_df):
    return int(pair_df.duplicated(["user_id", "pos_item", "neg_item"]).sum())


def count_reverse_conflicts(pair_df):
    forward_pairs = set(
        zip(pair_df["user_id"], pair_df["pos_item"], pair_df["neg_item"])
    )

    reverse_pairs = set(
        zip(pair_df["user_id"], pair_df["neg_item"], pair_df["pos_item"])
    )

    return int(len(forward_pairs & reverse_pairs))

### 1.4 Load configs

In [5]:
step1_config = read_json(step1_dir / "step1_config.json")
step2_config = read_json(step2_dir / "final_step2_config.json")
step2_pipeline_audit = read_json(step2_dir / "final_pipeline_audit.json")

random_seed = int(
    step1_config.get("random_seed", step1_config.get("seed", 42))
)

pair_min_diff = int(
    step1_config.get("pair_min_diff", step2_config.get("pair_min_diff", 2))
)

random.seed(random_seed)
np.random.seed(random_seed)

config_summary = pd.DataFrame([
    {
        "config_source": "step1_config",
        "dataset": step1_config.get("dataset", None),
        "pair_min_diff": step1_config.get("pair_min_diff", None),
        "train_min_pairs_per_user": step1_config.get("train_min_pairs_per_user", None),
        "train_max_pairs_per_user": step1_config.get("train_max_pairs_per_user", None),
        "pair_sampling": step1_config.get("pair_sampling", None),
        "random_seed": step1_config.get("random_seed", None),
    },
    {
        "config_source": "step2_config",
        "dataset": step2_config.get("dataset", None),
        "pair_min_diff": step2_config.get("pair_min_diff", None),
        "train_min_pairs_per_user": step2_config.get("min_pairs_per_user", None),
        "train_max_pairs_per_user": step2_config.get("max_pairs_per_user", None),
        "pair_sampling": step2_config.get("final_export_pair_sampling_scheme", None),
        "random_seed": step2_config.get("random_seed", None),
    },
])

display(config_summary)

,config_source,dataset,pair_min_diff,train_min_pairs_per_user,train_max_pairs_per_user,pair_sampling,random_seed
0,step1_config,MovieLens 1M,2,10,2000,proportional_stratified_by_rating_diff_with_us...,42.0
1,step2_config,NaN,2,10,2000,current_step1_gap2_stratified_cap,NaN


### 1.5 Load step 1 data

In [6]:
movies = load_parquet(step1_dir / "movies.parquet")
users = load_parquet(step1_dir / "users.parquet")
ratings_filtered = load_parquet(step1_dir / "ratings_filtered.parquet")

train_df = load_parquet(step1_dir / "train_split_model_universe.parquet")
validation_df = load_parquet(step1_dir / "validation_split_model_universe.parquet")
test_df = load_parquet(step1_dir / "test_split_model_universe.parquet")

if load_pair_splits:
    train_pairs = standardize_pair_columns(
        load_parquet(step1_dir / "train_pairs.parquet")
    )

    validation_pairs = standardize_pair_columns(
        load_parquet(step1_dir / "validation_pairs.parquet")
    )

    test_pairs = standardize_pair_columns(
        load_parquet(step1_dir / "test_pairs.parquet")
    )
else:
    train_pairs = None
    validation_pairs = None
    test_pairs = None

print("step 1 data loaded")

step 1 data loaded


### 1.6 Load step 2 embeddings and diagnostics

In [7]:
user_embeddings_df = load_parquet(step2_dir / "user_embeddings_final.parquet")
item_embeddings_df = load_parquet(step2_dir / "item_embeddings_final.parquet")

user_embeddings_npy = np.load(step2_dir / "user_embeddings_p_u.npy")
item_embeddings_npy = np.load(step2_dir / "item_embeddings_q_i.npy")
item_bias = np.load(step2_dir / "item_bias.npy")

user_index_mapping = load_parquet(step2_dir / "user_index_mapping.parquet")
item_index_mapping = load_parquet(step2_dir / "item_index_mapping.parquet")

final_pairwise_evaluation = load_parquet(
    step2_dir / "final_pairwise_evaluation.parquet"
)

final_generalization_summary = load_parquet(
    step2_dir / "final_generalization_summary.parquet"
)

embedding_diagnostics_summary = load_parquet(
    step2_dir / "embedding_diagnostics_summary.parquet"
)

print("step 2 data loaded")

step 2 data loaded


### 1.7 Identify embedding columns

In [8]:
user_embedding_columns = [
    column for column in user_embeddings_df.columns
    if column.startswith("p_")
]

item_embedding_columns = [
    column for column in item_embeddings_df.columns
    if column.startswith("q_")
]

if len(user_embedding_columns) == 0:
    raise ValueError("No user embedding columns found. Expected columns starting with 'p_'.")

if len(item_embedding_columns) == 0:
    raise ValueError("No item embedding columns found. Expected columns starting with 'q_'.")

user_embeddings_from_df = user_embeddings_df[
    user_embedding_columns
].to_numpy(dtype=np.float64)

item_embeddings_from_df = item_embeddings_df[
    item_embedding_columns
].to_numpy(dtype=np.float64)

print("number of user embedding columns:", len(user_embedding_columns))
print("first user embedding columns:", user_embedding_columns[:5])

print("number of item embedding columns:", len(item_embedding_columns))
print("first item embedding columns:", item_embedding_columns[:5])

number of user embedding columns: 64
first user embedding columns: ['p_000', 'p_001', 'p_002', 'p_003', 'p_004']
number of item embedding columns: 64
first item embedding columns: ['q_000', 'q_001', 'q_002', 'q_003', 'q_004']


### 1.8 Loaded object summary

In [9]:
loaded_object_rows = [
    {
        "object": "movies",
        "rows": len(movies),
        "columns": movies.shape[1],
        "users": None,
        "items": movies["movie_id"].nunique(),
    },
    {
        "object": "users",
        "rows": len(users),
        "columns": users.shape[1],
        "users": users["user_id"].nunique(),
        "items": None,
    },
    {
        "object": "ratings_filtered",
        "rows": len(ratings_filtered),
        "columns": ratings_filtered.shape[1],
        "users": ratings_filtered["user_id"].nunique(),
        "items": ratings_filtered["movie_id"].nunique(),
    },
    {
        "object": "train_df",
        "rows": len(train_df),
        "columns": train_df.shape[1],
        "users": train_df["user_id"].nunique(),
        "items": train_df["movie_id"].nunique(),
    },
    {
        "object": "validation_df",
        "rows": len(validation_df),
        "columns": validation_df.shape[1],
        "users": validation_df["user_id"].nunique(),
        "items": validation_df["movie_id"].nunique(),
    },
    {
        "object": "test_df",
        "rows": len(test_df),
        "columns": test_df.shape[1],
        "users": test_df["user_id"].nunique(),
        "items": test_df["movie_id"].nunique(),
    },
    {
        "object": "user_embeddings_df",
        "rows": len(user_embeddings_df),
        "columns": user_embeddings_df.shape[1],
        "users": user_embeddings_df["user_id"].nunique(),
        "items": None,
    },
    {
        "object": "item_embeddings_df",
        "rows": len(item_embeddings_df),
        "columns": item_embeddings_df.shape[1],
        "users": None,
        "items": item_embeddings_df["movie_id"].nunique(),
    },
]

if load_pair_splits:
    for split_name, pair_df in [
        ("train_pairs", train_pairs),
        ("validation_pairs", validation_pairs),
        ("test_pairs", test_pairs),
    ]:
        loaded_object_rows.append({
            "object": split_name,
            "rows": len(pair_df),
            "columns": pair_df.shape[1],
            "users": pair_df["user_id"].nunique(),
            "items": len(
                set(pair_df["pos_item"].unique())
                | set(pair_df["neg_item"].unique())
            ),
        })

loaded_object_summary = pd.DataFrame(loaded_object_rows)

display(loaded_object_summary)

,object,rows,columns,users,items
0,movies,3883,3,NaN,3883.0
1,users,6040,5,6040.0,NaN
2,ratings_filtered,960916,5,4726.0,3250.0
3,train_df,759306,5,4722.0,3250.0
4,validation_df,100787,5,4722.0,3204.0
5,test_df,100564,5,4722.0,3226.0
6,user_embeddings_df,4722,70,4722.0,NaN
7,item_embeddings_df,3250,69,NaN,3250.0
8,train_pairs,5692244,8,4722.0,3250.0
9,validation_pairs,478098,8,4158.0,3193.0


### 1.9 Embedding integrity checks

In [10]:
expected_user_count = int(step2_config.get("n_users", len(user_index_mapping)))
expected_item_count = int(step2_config.get("n_items", len(item_index_mapping)))
expected_embedding_dim = int(
    step2_config.get("embedding_dim", len(user_embedding_columns))
)

embedding_integrity_rows = [
    {
        "check": "user parquet rows == user mapping rows",
        "result": len(user_embeddings_df) == len(user_index_mapping),
        "left": len(user_embeddings_df),
        "right": len(user_index_mapping),
    },
    {
        "check": "item parquet rows == item mapping rows",
        "result": len(item_embeddings_df) == len(item_index_mapping),
        "left": len(item_embeddings_df),
        "right": len(item_index_mapping),
    },
    {
        "check": "user npy rows == user mapping rows",
        "result": user_embeddings_npy.shape[0] == len(user_index_mapping),
        "left": user_embeddings_npy.shape[0],
        "right": len(user_index_mapping),
    },
    {
        "check": "item npy rows == item mapping rows",
        "result": item_embeddings_npy.shape[0] == len(item_index_mapping),
        "left": item_embeddings_npy.shape[0],
        "right": len(item_index_mapping),
    },
    {
        "check": "item bias rows == item mapping rows",
        "result": item_bias.shape[0] == len(item_index_mapping),
        "left": item_bias.shape[0],
        "right": len(item_index_mapping),
    },
    {
        "check": "user embedding dim == config embedding_dim",
        "result": len(user_embedding_columns) == expected_embedding_dim,
        "left": len(user_embedding_columns),
        "right": expected_embedding_dim,
    },
    {
        "check": "item embedding dim == config embedding_dim",
        "result": len(item_embedding_columns) == expected_embedding_dim,
        "left": len(item_embedding_columns),
        "right": expected_embedding_dim,
    },
    {
        "check": "config user count matches loaded users",
        "result": expected_user_count == len(user_index_mapping),
        "left": expected_user_count,
        "right": len(user_index_mapping),
    },
    {
        "check": "config item count matches loaded items",
        "result": expected_item_count == len(item_index_mapping),
        "left": expected_item_count,
        "right": len(item_index_mapping),
    },
    {
        "check": "user parquet embeddings match user npy",
        "result": bool(np.allclose(user_embeddings_from_df, user_embeddings_npy, atol=1e-7)),
        "left": "parquet",
        "right": "npy",
    },
    {
        "check": "item parquet embeddings match item npy",
        "result": bool(np.allclose(item_embeddings_from_df, item_embeddings_npy, atol=1e-7)),
        "left": "parquet",
        "right": "npy",
    },
    {
        "check": "no nan in user embeddings",
        "result": not bool(np.isnan(user_embeddings_from_df).any()),
        "left": int(np.isnan(user_embeddings_from_df).sum()),
        "right": 0,
    },
    {
        "check": "no nan in item embeddings",
        "result": not bool(np.isnan(item_embeddings_from_df).any()),
        "left": int(np.isnan(item_embeddings_from_df).sum()),
        "right": 0,
    },
    {
        "check": "no inf in user embeddings",
        "result": not bool(np.isinf(user_embeddings_from_df).any()),
        "left": int(np.isinf(user_embeddings_from_df).sum()),
        "right": 0,
    },
    {
        "check": "no inf in item embeddings",
        "result": not bool(np.isinf(item_embeddings_from_df).any()),
        "left": int(np.isinf(item_embeddings_from_df).sum()),
        "right": 0,
    },
]

embedding_integrity_summary = pd.DataFrame(embedding_integrity_rows)

display(embedding_integrity_summary)

if not embedding_integrity_summary["result"].all():
    failed_checks = embedding_integrity_summary[
        ~embedding_integrity_summary["result"]
    ]

    raise AssertionError(
        "Some embedding integrity checks failed:\n"
        + failed_checks.to_string(index=False)
    )

,check,result,left,right
0,user parquet rows == user mapping rows,True,4722,4722
1,item parquet rows == item mapping rows,True,3250,3250
2,user npy rows == user mapping rows,True,4722,4722
3,item npy rows == item mapping rows,True,3250,3250
4,item bias rows == item mapping rows,True,3250,3250
5,user embedding dim == config embedding_dim,True,64,64
6,item embedding dim == config embedding_dim,True,64,64
7,config user count matches loaded users,True,4722,4722
8,config item count matches loaded items,True,3250,3250
9,user parquet embeddings match user npy,True,parquet,npy


### 1.10 Universe checks

In [11]:
embedding_user_ids = set(user_embeddings_df["user_id"])
embedding_movie_ids = set(item_embeddings_df["movie_id"])

train_user_ids = set(train_df["user_id"])
validation_user_ids = set(validation_df["user_id"])
test_user_ids = set(test_df["user_id"])

train_movie_ids = set(train_df["movie_id"])
validation_movie_ids = set(validation_df["movie_id"])
test_movie_ids = set(test_df["movie_id"])

universe_check_rows = [
    {
        "check": "train users subset of user embeddings",
        "result": train_user_ids.issubset(embedding_user_ids),
        "left_count": len(train_user_ids),
        "right_count": len(embedding_user_ids),
        "difference_count": len(train_user_ids - embedding_user_ids),
    },
    {
        "check": "validation users subset of user embeddings",
        "result": validation_user_ids.issubset(embedding_user_ids),
        "left_count": len(validation_user_ids),
        "right_count": len(embedding_user_ids),
        "difference_count": len(validation_user_ids - embedding_user_ids),
    },
    {
        "check": "test users subset of user embeddings",
        "result": test_user_ids.issubset(embedding_user_ids),
        "left_count": len(test_user_ids),
        "right_count": len(embedding_user_ids),
        "difference_count": len(test_user_ids - embedding_user_ids),
    },
    {
        "check": "train items subset of item embeddings",
        "result": train_movie_ids.issubset(embedding_movie_ids),
        "left_count": len(train_movie_ids),
        "right_count": len(embedding_movie_ids),
        "difference_count": len(train_movie_ids - embedding_movie_ids),
    },
    {
        "check": "validation items subset of item embeddings",
        "result": validation_movie_ids.issubset(embedding_movie_ids),
        "left_count": len(validation_movie_ids),
        "right_count": len(embedding_movie_ids),
        "difference_count": len(validation_movie_ids - embedding_movie_ids),
    },
    {
        "check": "test items subset of item embeddings",
        "result": test_movie_ids.issubset(embedding_movie_ids),
        "left_count": len(test_movie_ids),
        "right_count": len(embedding_movie_ids),
        "difference_count": len(test_movie_ids - embedding_movie_ids),
    },
]

if load_pair_splits:
    for split_name, pair_df in [
        ("train_pairs", train_pairs),
        ("validation_pairs", validation_pairs),
        ("test_pairs", test_pairs),
    ]:
        pair_user_ids = set(pair_df["user_id"])
        pair_movie_ids = set(pair_df["pos_item"]) | set(pair_df["neg_item"])

        universe_check_rows += [
            {
                "check": f"{split_name} users subset of user embeddings",
                "result": pair_user_ids.issubset(embedding_user_ids),
                "left_count": len(pair_user_ids),
                "right_count": len(embedding_user_ids),
                "difference_count": len(pair_user_ids - embedding_user_ids),
            },
            {
                "check": f"{split_name} items subset of item embeddings",
                "result": pair_movie_ids.issubset(embedding_movie_ids),
                "left_count": len(pair_movie_ids),
                "right_count": len(embedding_movie_ids),
                "difference_count": len(pair_movie_ids - embedding_movie_ids),
            },
            {
                "check": f"{split_name} all rating_diff >= pair_min_diff",
                "result": bool(pair_df["rating_diff"].ge(pair_min_diff).all()),
                "left_count": int(pair_df["rating_diff"].min()),
                "right_count": pair_min_diff,
                "difference_count": int((pair_df["rating_diff"] < pair_min_diff).sum()),
            },
            {
                "check": f"{split_name} no same pos and neg item",
                "result": bool((pair_df["pos_item"] != pair_df["neg_item"]).all()),
                "left_count": int((pair_df["pos_item"] == pair_df["neg_item"]).sum()),
                "right_count": 0,
                "difference_count": int((pair_df["pos_item"] == pair_df["neg_item"]).sum()),
            },
            {
                "check": f"{split_name} no duplicate pair rows",
                "result": count_semantic_duplicates(pair_df) == 0,
                "left_count": count_semantic_duplicates(pair_df),
                "right_count": 0,
                "difference_count": count_semantic_duplicates(pair_df),
            },
            {
                "check": f"{split_name} no reverse conflicts",
                "result": count_reverse_conflicts(pair_df) == 0,
                "left_count": count_reverse_conflicts(pair_df),
                "right_count": 0,
                "difference_count": count_reverse_conflicts(pair_df),
            },
        ]

universe_check_summary = pd.DataFrame(universe_check_rows)

display(universe_check_summary)

if not universe_check_summary["result"].all():
    failed_checks = universe_check_summary[
        ~universe_check_summary["result"]
    ]

    raise AssertionError(
        "Some universe checks failed:\n"
        + failed_checks.to_string(index=False)
    )

,check,result,left_count,right_count,difference_count
0,train users subset of user embeddings,True,4722,4722,0
1,validation users subset of user embeddings,True,4722,4722,0
2,test users subset of user embeddings,True,4722,4722,0
3,train items subset of item embeddings,True,3250,3250,0
4,validation items subset of item embeddings,True,3204,3250,0
5,test items subset of item embeddings,True,3226,3250,0
6,train_pairs users subset of user embeddings,True,4722,4722,0
7,train_pairs items subset of item embeddings,True,3250,3250,0
8,train_pairs all rating_diff >= pair_min_diff,True,2,2,0
9,train_pairs no same pos and neg item,True,0,0,0


### 1.11 Embedding numeric diagnostics

In [12]:
embedding_numeric_check_summary = pd.DataFrame([
    make_matrix_check("user_embeddings_from_df", user_embeddings_from_df),
    make_matrix_check("item_embeddings_from_df", item_embeddings_from_df),
    make_matrix_check("user_embeddings_npy", user_embeddings_npy),
    make_matrix_check("item_embeddings_npy", item_embeddings_npy),
    {
        "name": "item_bias",
        "shape": tuple(item_bias.shape),
        "dtype": str(item_bias.dtype),
        "has_nan": bool(np.isnan(item_bias).any()),
        "has_inf": bool(np.isinf(item_bias).any()),
        "min": float(np.min(item_bias)),
        "max": float(np.max(item_bias)),
        "mean": float(np.mean(item_bias)),
        "std": float(np.std(item_bias)),
    },
])

display(embedding_numeric_check_summary.round(6))

embedding_norm_summary = pd.DataFrame([
    make_norm_summary("user_embeddings_p_u", user_embeddings_from_df),
    make_norm_summary("item_embeddings_q_i", item_embeddings_from_df),
])

display(embedding_norm_summary.round(6))

print("embedding diagnostics from step 2:")
display(embedding_diagnostics_summary.round(6))

,name,shape,dtype,has_nan,has_inf,min,max,mean,std
0,user_embeddings_from_df,"(4722, 64)",float64,False,False,-0.986428,0.936054,0.010254,0.249333
1,item_embeddings_from_df,"(3250, 64)",float64,False,False,-1.138831,1.172526,-0.003116,0.255475
2,user_embeddings_npy,"(4722, 64)",float32,False,False,-0.986428,0.936054,0.010254,0.249333
3,item_embeddings_npy,"(3250, 64)",float32,False,False,-1.138831,1.172526,-0.003116,0.255475
4,item_bias,"(3250,)",float32,False,False,-0.514834,0.340417,-0.028880,0.134756


,embedding,rows,dimensions,mean_l2_norm,median_l2_norm,std_l2_norm,min_l2_norm,p01_l2_norm,p05_l2_norm,p95_l2_norm,p99_l2_norm,max_l2_norm,zero_norm_rows
0,user_embeddings_p_u,4722,64,1.951730,1.969063,0.419726,0.753706,1.032941,1.242593,2.626891,2.860671,3.274429,0
1,item_embeddings_q_i,3250,64,1.943475,1.864486,0.632968,0.291399,0.783628,1.025713,3.104769,3.666101,4.527428,0


embedding diagnostics from step 2:


,embedding_type,rows,dimensions,mean_value,std_value,min_value,max_value,mean_l2_norm,median_l2_norm,min_l2_norm,max_l2_norm
0,user_embeddings_p_u,4722,64,0.010254,0.249333,-0.986428,0.936054,1.951730,1.969063,0.753706,3.274429
1,item_embeddings_q_i,3250,64,-0.003116,0.255475,-1.138831,1.172526,1.943475,1.864486,0.291399,4.527428


### 1.12 Preview key data

In [13]:
print("movies preview")
display(movies.head())

print("users preview")
display(users.head())

print("user embeddings preview")
display(user_embeddings_df.head())

print("item embeddings preview")
display(item_embeddings_df.head())

print("final pairwise evaluation")
display(final_pairwise_evaluation)

print("final generalization summary")
display(final_generalization_summary)

movies preview


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


users preview


,user_id,gender,age,occupation,zip_code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


user embeddings preview


...

,item_idx,movie_id,q_000,q_001,q_002,q_003,q_004,q_005,q_006,q_007,...,q_057,q_058,q_059,q_060,q_061,q_062,q_063,item_bias,title,genres
0,0,1,0.073422,-0.165789,0.436513,0.610143,-0.184806,0.321441,0.468652,-0.453784,...,0.180348,0.413844,-0.838689,-0.138529,-0.084099,-0.098945,-0.278987,0.170157,Toy Story (1995),Animation|Children's|Comedy
1,1,2,0.544998,-0.092697,-0.159939,-0.187868,-0.222771,-0.390883,0.120806,-0.017216,...,0.023442,0.141803,-0.102082,-0.399405,-0.010345,0.244667,-0.030912,-0.019771,Jumanji (1995),Adventure|Children's|Fantasy
2,2,3,0.210633,0.223043,-0.065718,0.035314,0.276970,-0.592555,-0.239138,0.078779,...,-0.394569,0.091260,0.057280,-0.412541,0.087471,0.128797,-0.179712,-0.089849,Grumpier Old Men (1995),Comedy|Romance
3,3,4,-0.045509,0.395397,-0.296207,-0.218395,-0.358650,-0.145529,-0.179090,0.076830,...,-0.308712,0.150024,0.135028,0.099097,-0.074686,0.343203,-0.064170,-0.180413,Waiting to Exhale (1995),Comedy|Drama
4,4,5,0.350826,0.236962,-0.125356,-0.165884,0.132370,-0.242794,0.291093,0.132021,...,-0.316677,0.290306,-0.132672,-0.325404,-0.064859,0.095226,-0.282564,-0.014695,Father of the Bride Part II (1995),Comedy


final pairwise evaluation


,split,pairs,users,comparison_accuracy_micro,comparison_accuracy_macro_user,roc_auc_symmetric,brier_positive_pairs,log_loss_positive_pairs,mean_score_diff,median_score_diff
0,train_final_step1_pairs,5692244,4722,0.939408,0.948321,0.985213,0.050833,0.184788,2.899384,2.685098
1,validation_pairs,478098,4158,0.808433,0.757608,0.896024,0.131186,0.404075,1.903471,1.559963
2,test_pairs,507550,4164,0.817919,0.774139,0.903074,0.126110,0.394358,2.023247,1.721708


final generalization summary


,metric,value
0,train_macro_accuracy,0.948321
1,validation_macro_accuracy,0.757608
2,test_macro_accuracy,0.774139
3,train_minus_validation_macro_gap,0.190713
4,validation_minus_test_macro_gap,-0.016531
5,validation_auc,0.896024
6,test_auc,0.903074
